# 07 — RL agent (walk 1, PPO + cost-variant sweep)

PPO policy that allocates weights over the walk-1 ranker top-30 each Friday.
Trains separate policies at 4 transaction-cost levels (5/2/10/20 bps) so one
overnight run produces all the cost-sensitivity ablations for the paper.

**Spec:** `docs/superpowers/specs/2026-05-17-rl-agent-design.md`.

SB3 conventions: `check_env` validation → `EvalCallback` (best by val) →
`VecNormalize` (saved for backtest reuse) → TensorBoard. Single-walk MVP;
walks 2-16 deferred. Backtest = notebook 08.

## A. Setup

In [ ]:
from __future__ import annotations
import json, time
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import gymnasium as gym
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.callbacks import (
    EvalCallback, CheckpointCallback, ProgressBarCallback,
)

from src.utils.io import processed_dir, repo_root
from src.utils.ranker import friday_only, project_text_to_pca, load_walk_pca
from src.utils.rl_env import (
    PortfolioEnv,
    build_scoreboard_from_scored_panel,
)

WALK_ID = 1
TRAIN_START, TRAIN_END = '2002-01-01', '2007-12-31'
VAL_START,   VAL_END   = '2008-01-01', '2008-12-31'

PANEL_DIR  = processed_dir() / 'training_panel'
EMBED_DIR  = processed_dir() / 'finbert_stockday_embed'
RANKER_DIR = repo_root() / 'artifacts' / 'ranker' / f'walk-{WALK_ID:03d}'
OUT_ROOT   = repo_root() / 'artifacts' / 'rl' / f'walk-{WALK_ID:03d}'
OUT_ROOT.mkdir(parents=True, exist_ok=True)

# Cost sweep — primary first; resumable across costs.
COSTS_BPS = [5, 2, 10, 20]
TOP_K = 30
EPISODE_LENGTH = 52
MAX_WEIGHT = 1.0  # no per-stock cap; let policy concentrate if it wants
TOTAL_TIMESTEPS = 2_000_000
EVAL_FREQ = 10_000
CHECKPOINT_FREQ = 200_000
N_ENVS = 4
RANDOM_STATE = 42

# Reward design (locked from autoresearch session, best config = EXP-023).
# See experiments/STATE.md for full HP search log.
REWARD_TYPE = 'sharpe'
SHARPE_WINDOW = 16
ACTION_HIGH = 5.0

print(f'walk {WALK_ID}; train {TRAIN_START}..{TRAIN_END}, val {VAL_START}..{VAL_END}')
print(f'cost sweep: {COSTS_BPS} bps; {TOTAL_TIMESTEPS:,} timesteps per variant')
print(f'reward: {REWARD_TYPE} (sharpe_window={SHARPE_WINDOW}); action_high={ACTION_HIGH}')
print(f'out_root: {OUT_ROOT.relative_to(repo_root())}')

## B. Build the scoreboard

For each Friday in `[train_start, val_end]`, run the walk-1 ranker on every
stock in the universe, keep the top-30 by score. Persist to a parquet so
re-runs / kernel restarts don't re-score the panel.

In [ ]:
SCOREBOARD_PATH = OUT_ROOT / 'scoreboard.parquet'

if SCOREBOARD_PATH.exists():
    scoreboard = pd.read_parquet(SCOREBOARD_PATH)
    print(f'loaded scoreboard from disk: {len(scoreboard)} rows, '
          f'{scoreboard["date"].nunique()} Fridays')
else:
    ranker_bundle = joblib.load(RANKER_DIR / 'model.joblib')
    ranker_model = ranker_bundle['model']
    ranker_features = ranker_bundle['feature_names']
    pca, n_pca = load_walk_pca(WALK_ID)
    print(f'ranker loaded: {len(ranker_features)} features; PCA n_components={n_pca}')

    def _load_panel(start: str, end: str) -> pd.DataFrame:
        s, e = pd.Timestamp(start), pd.Timestamp(end)
        frames = []
        for y in range(s.year, e.year + 1):
            for p in sorted((PANEL_DIR / f'year={y}').glob('*.parquet')):
                df = pd.read_parquet(p)
                df['date'] = pd.to_datetime(df['date'])
                df = df[(df['date'] >= s) & (df['date'] <= e)]
                frames.append(df)
        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    def _load_embed(start: str, end: str) -> pd.DataFrame:
        s, e = pd.Timestamp(start), pd.Timestamp(end)
        frames = []
        for y in range(s.year, e.year + 1):
            for p in sorted((EMBED_DIR / f'year={y}').glob('*.parquet')):
                df = pd.read_parquet(p, columns=['permno', 'date', 'vec'])
                df['date'] = pd.to_datetime(df['date'])
                df = df[(df['date'] >= s) & (df['date'] <= e)]
                frames.append(df)
        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    panel = _load_panel(TRAIN_START, VAL_END)
    embed = _load_embed(TRAIN_START, VAL_END)
    embed_pca = project_text_to_pca(embed, pca)

    # Friday-only + join PCA + drop rows w/o a forward return.
    fri = friday_only(panel).merge(embed_pca, on=['permno', 'date'], how='inner')
    fri = fri.dropna(subset=['fwd_ret_5d']).copy()

    # Build the feature matrix in the same column order the ranker expects.
    # Missing cols (shouldn't happen, but defensive) get NaN.
    X = pd.DataFrame({c: fri[c] if c in fri.columns else np.nan for c in ranker_features})
    fri['score'] = ranker_model.predict(X)

    scoreboard = build_scoreboard_from_scored_panel(fri, top_k=TOP_K)
    scoreboard.to_parquet(SCOREBOARD_PATH, compression='zstd', index=False)
    print(f'wrote scoreboard: {len(scoreboard)} rows, '
          f'{scoreboard["date"].nunique()} Fridays -> '
          f'{SCOREBOARD_PATH.relative_to(repo_root())}')

# Split into train / val views once.
scoreboard['date'] = pd.to_datetime(scoreboard['date'])
scoreboard_train = scoreboard[(scoreboard['date'] >= TRAIN_START) & (scoreboard['date'] <= TRAIN_END)].copy()
scoreboard_val   = scoreboard[(scoreboard['date'] >= VAL_START)   & (scoreboard['date'] <= VAL_END)].copy()
print(f'train Fridays: {scoreboard_train["date"].nunique()}; '
      f'val Fridays: {scoreboard_val["date"].nunique()}')

## C. `train_one(cost_bps, out_dir)` — single cost-variant training

Builds train (4-env DummyVecEnv, randomized starts) and val (1-env, deterministic)
envs at the given cost level, runs PPO. `EvalCallback` saves the best policy
by val mean reward; `CheckpointCallback` writes rolling backups; TensorBoard
logs reward + value loss + entropy.

`VecNormalize` stats are saved alongside the policy so notebook 08's backtest
applies the same observation normalization to 2009 data.

In [ ]:
def _make_env_fn(scoreboard_subset: pd.DataFrame, cost_bps: float, seed: int):
    def _thunk():
        env = PortfolioEnv(
            scoreboard=scoreboard_subset,
            top_k=TOP_K,
            episode_length=EPISODE_LENGTH,
            cost_bps=cost_bps,
            max_weight=MAX_WEIGHT,
            reward_type=REWARD_TYPE,
            sharpe_window=SHARPE_WINDOW,
            action_high=ACTION_HIGH,
        )
        env.reset(seed=seed)
        return Monitor(env)  # SB3 episode-stats logger; silences the eval warning
    return _thunk


def train_one(cost_bps: float, out_dir: Path) -> dict:
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / 'ckpts').mkdir(exist_ok=True)
    (out_dir / 'tb').mkdir(exist_ok=True)

    train_vec = DummyVecEnv([
        _make_env_fn(scoreboard_train, cost_bps, RANDOM_STATE + i)
        for i in range(N_ENVS)
    ])
    train_vec = VecNormalize(train_vec, norm_obs=True, norm_reward=False, clip_obs=10.0)

    val_vec = DummyVecEnv([_make_env_fn(scoreboard_val, cost_bps, RANDOM_STATE + 1000)])
    val_vec = VecNormalize(val_vec, norm_obs=True, norm_reward=False, clip_obs=10.0,
                           training=False)

    eval_cb = EvalCallback(
        val_vec,
        best_model_save_path=str(out_dir),
        log_path=str(out_dir),
        eval_freq=max(EVAL_FREQ // N_ENVS, 1),
        n_eval_episodes=1,
        deterministic=True,
    )
    ckpt_cb = CheckpointCallback(
        save_freq=max(CHECKPOINT_FREQ // N_ENVS, 1),
        save_path=str(out_dir / 'ckpts'),
        name_prefix='ppo',
    )

    # HPs locked from autoresearch session (best = EXP-023).
    # Search log: experiments/STATE.md + experiments/results.tsv.
    # Key findings:
    #   - reward_type='sharpe' (rolling Sharpe of excess return) >> 'excess_return'
    #     (Sharpe 2.07 vs 1.52 at 1M timesteps)
    #   - sharpe_window=16 weeks > 8 or 26 (Sharpe 2.19 vs 2.07/2.13)
    #   - ent_coef=0.02 best (Sharpe 2.23, monotonic improvement from 0.0001 → 0.02)
    #   - 2M timesteps ≈ 1M with these HPs (early convergence)
    #   - SAC, wider net, score-bias projection, alternative rewards all worse
    model = PPO(
        'MlpPolicy',
        train_vec,
        policy_kwargs=dict(net_arch=[128, 64]),
        learning_rate=1e-4,
        n_steps=2048,
        batch_size=64,
        n_epochs=5,
        gamma=0.99,
        gae_lambda=0.95,
        clip_range=0.15,
        ent_coef=0.02,
        vf_coef=0.5,
        max_grad_norm=0.5,
        target_kl=0.03,
        device='cpu',
        tensorboard_log=str(out_dir / 'tb'),
        seed=RANDOM_STATE,
        verbose=0,
    )

    t0 = time.time()
    model.learn(
        total_timesteps=TOTAL_TIMESTEPS,
        callback=[eval_cb, ckpt_cb, ProgressBarCallback()],
        progress_bar=False,  # ProgressBarCallback handles it
    )
    elapsed = time.time() - t0

    model.save(out_dir / 'final_policy')
    train_vec.save(str(out_dir / 'vec_normalize.pkl'))

    metrics = {
        'cost_bps': cost_bps,
        'total_timesteps': TOTAL_TIMESTEPS,
        'n_envs': N_ENVS,
        'wall_time_sec': elapsed,
        'wall_time_hr': elapsed / 3600.0,
        'best_val_mean_reward': float(eval_cb.best_mean_reward),
    }
    (out_dir / 'training_metrics.json').write_text(json.dumps(metrics, indent=2))
    print(f'  cost={cost_bps}bps done; wall={elapsed/60:.1f} min; '
          f'best_val_mean_reward={metrics["best_val_mean_reward"]:.4f}')
    return metrics

## D. Train all cost variants

Resumable: any cost variant whose `final_policy.zip` already exists is skipped.
Long-running — designed to be kicked off and left overnight.

In [ ]:
all_metrics = []
for cost_bps in COSTS_BPS:
    out_dir = OUT_ROOT / f'cost-{cost_bps:03d}bps'
    final_path = out_dir / 'final_policy.zip'
    if final_path.exists():
        existing = json.loads((out_dir / 'training_metrics.json').read_text())
        print(f'cost={cost_bps}bps: exists '
              f'(best_val_mean_reward={existing.get("best_val_mean_reward", float("nan")):.4f}) — skipping')
        all_metrics.append(existing)
        continue
    print(f'\n=== training cost={cost_bps}bps -> {out_dir.relative_to(repo_root())} ===')
    m = train_one(float(cost_bps), out_dir)
    all_metrics.append(m)

(OUT_ROOT / 'all_costs_summary.json').write_text(
    json.dumps({'cost_variants': all_metrics}, indent=2)
)
print(f'\nfinished {len(all_metrics)} cost variants; '
      f'summary -> {(OUT_ROOT / "all_costs_summary.json").relative_to(repo_root())}')

## E. Cross-cost diagnostics

Per-cost summary table + bar plot of best val mean reward. Intuitive read on
whether the policy's quality degrades as transaction costs rise (the expected
direction). True comparison vs baselines happens in notebook 08.

In [ ]:
import matplotlib.pyplot as plt

summary = pd.DataFrame(all_metrics).sort_values('cost_bps').reset_index(drop=True)
display_cols = ['cost_bps', 'total_timesteps', 'wall_time_hr', 'best_val_mean_reward']
print('per-cost-variant summary:')
print(summary[display_cols].to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(summary['cost_bps'].astype(str) + ' bps',
       summary['best_val_mean_reward'], color='steelblue')
ax.axhline(0, color='black', lw=0.5)
ax.set_xlabel('transaction cost')
ax.set_ylabel('best val mean reward (1-year episode)')
ax.set_title('PPO val reward across cost regimes (walk 1)')
plt.tight_layout()
plt.show()

## F. Multi-walk extension (walks 2-16, 5 bps only)

Run AFTER walk-1's cost sweep has produced acceptable best policies. For each
walk N=2..16:
1. Build that walk's scoreboard from its own ranker (walk-N ranker) on the
   walk-N train (2002 → 2007+N-1) and val (2008+N-1) windows.
2. Train PPO at 5 bps using the frozen HPs established in walk-1.
3. Save to `artifacts/rl/walk-{N:03d}/cost-005bps/`.

Resumable. Designed to be kicked off overnight (~12-14 hours for 15 walks).
Output of this cell + cells A-E feeds the full-period backtest in notebook 08.

In [ ]:
MULTI_WALK_COST_BPS = 5  # single cost level for multi-walk extension
MULTI_WALK_RANGE = range(2, 17)  # walks 2..16


def _build_walk_scoreboard(walk_id: int, train_start: str, val_end: str) -> pd.DataFrame:
    """Score panel Fridays in [train_start, val_end] with walk-N's ranker + PCA."""
    rdir = repo_root() / 'artifacts' / 'ranker' / f'walk-{walk_id:03d}'
    ranker_bundle = joblib.load(rdir / 'model.joblib')
    ranker_model = ranker_bundle['model']
    ranker_features = ranker_bundle['feature_names']
    pca, _ = load_walk_pca(walk_id)

    def _load(dir_, start, end, cols=None):
        s, e = pd.Timestamp(start), pd.Timestamp(end)
        frames = []
        for y in range(s.year, e.year + 1):
            for p in sorted((dir_ / f'year={y}').glob('*.parquet')):
                df = pd.read_parquet(p, columns=cols)
                df['date'] = pd.to_datetime(df['date'])
                df = df[(df['date'] >= s) & (df['date'] <= e)]
                frames.append(df)
        return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

    panel = _load(PANEL_DIR, train_start, val_end)
    embed = _load(EMBED_DIR, train_start, val_end, cols=['permno', 'date', 'vec'])
    embed_pca = project_text_to_pca(embed, pca)
    fri = friday_only(panel).merge(embed_pca, on=['permno', 'date'], how='inner')
    fri = fri.dropna(subset=['fwd_ret_5d']).copy()

    X = pd.DataFrame({c: fri[c] if c in fri.columns else np.nan for c in ranker_features})
    fri['score'] = ranker_model.predict(X)
    return build_scoreboard_from_scored_panel(fri, top_k=TOP_K)


def train_walk_n(walk_id: int) -> dict:
    """Build scoreboard for walk N, train PPO at MULTI_WALK_COST_BPS, persist."""
    out_dir = repo_root() / 'artifacts' / 'rl' / f'walk-{walk_id:03d}' / f'cost-{MULTI_WALK_COST_BPS:03d}bps'
    final_path = out_dir / 'final_policy.zip'
    if final_path.exists():
        existing = json.loads((out_dir / 'training_metrics.json').read_text())
        print(f'walk {walk_id}: exists '
              f'(best_val_mean_reward={existing.get("best_val_mean_reward", float("nan")):.4f}) — skipping')
        return existing

    # Walk N: train 2002 → 2007+N-1, val 2008+N-1.
    train_end_year = 2007 + walk_id - 1
    val_year = train_end_year + 1
    walk_train_start, walk_train_end = '2002-01-01', f'{train_end_year}-12-31'
    walk_val_start,   walk_val_end   = f'{val_year}-01-01', f'{val_year}-12-31'

    # Scoreboard cache per walk (avoids re-scoring if we resume).
    walk_dir = repo_root() / 'artifacts' / 'rl' / f'walk-{walk_id:03d}'
    walk_dir.mkdir(parents=True, exist_ok=True)
    sb_path = walk_dir / 'scoreboard.parquet'
    if sb_path.exists():
        sb = pd.read_parquet(sb_path)
        sb['date'] = pd.to_datetime(sb['date'])
        print(f'  walk {walk_id}: loaded cached scoreboard ({len(sb)} rows)')
    else:
        print(f'  walk {walk_id}: building scoreboard ({walk_train_start}..{walk_val_end})...')
        sb = _build_walk_scoreboard(walk_id, walk_train_start, walk_val_end)
        sb.to_parquet(sb_path, compression='zstd', index=False)
        print(f'  walk {walk_id}: scoreboard built ({len(sb)} rows)')

    sb['date'] = pd.to_datetime(sb['date'])
    sb_train = sb[(sb['date'] >= walk_train_start) & (sb['date'] <= walk_train_end)].copy()
    sb_val   = sb[(sb['date'] >= walk_val_start)   & (sb['date'] <= walk_val_end)].copy()
    print(f'  walk {walk_id}: train Fridays={sb_train["date"].nunique()}, '
          f'val Fridays={sb_val["date"].nunique()}')

    # Temporarily swap globals used by train_one (cell C uses scoreboard_train / scoreboard_val).
    global scoreboard_train, scoreboard_val
    save_train, save_val = scoreboard_train, scoreboard_val
    scoreboard_train, scoreboard_val = sb_train, sb_val
    try:
        m = train_one(float(MULTI_WALK_COST_BPS), out_dir)
        m['walk_id'] = walk_id
        return m
    finally:
        scoreboard_train, scoreboard_val = save_train, save_val


multi_walk_metrics = []
for w in MULTI_WALK_RANGE:
    print(f'\n=== walk {w} ===')
    m = train_walk_n(w)
    multi_walk_metrics.append(m)

(repo_root() / 'artifacts' / 'rl' / 'multi_walk_summary.json').write_text(
    json.dumps({'walks': multi_walk_metrics}, indent=2)
)
print(f'\nfinished walks 2-16; summary -> artifacts/rl/multi_walk_summary.json')